In [1]:
!nvidia-smi

Mon Sep  7 19:32:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!git clone https://github.com/yeezyyoba/amharic-sentiment-analysis.git
%cd amharic-sentiment-analysis

Cloning into 'amharic-sentiment-analysis'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 55 (delta 13), reused 39 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 330.05 KiB | 15.00 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/amharic-sentiment-analysis


In [4]:
!pip install transformers datasets evaluate accelerate torch scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [5]:
import torch
import transformers
import datasets
import evaluate

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
GPU available: True
GPU name: Tesla T4


In [6]:
# ── Imports ────────────────────────────────────────────────────
import os
import json
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate

# ── Config ─────────────────────────────────────────────────────
MODEL_NAME = "Davlan/afro-xlmr-base"  # base for speed, large if time allows
MAX_LENGTH = 72
SEEDS = [42, 123, 456]
NUM_EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
DRIVE_SAVE_PATH = "/content/drive/MyDrive/amharic_sentiment_models"

# Label maps
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f"Model: {MODEL_NAME}")
print(f"Seeds: {SEEDS}")
print(f"Save path: {DRIVE_SAVE_PATH}")

Model: Davlan/afro-xlmr-base
Seeds: [42, 123, 456]
Save path: /content/drive/MyDrive/amharic_sentiment_models


In [9]:
!git pull origin feat/day-7-transformer

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 11 (delta 3), reused 11 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 2.37 MiB | 7.05 MiB/s, done.
From https://github.com/yeezyyoba/amharic-sentiment-analysis
 * branch            feat/day-7-transformer -> FETCH_HEAD
 * [new branch]      feat/day-7-transformer -> origin/feat/day-7-transformer
hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-o

In [12]:
!git pull origin main
!ls data/processed/

remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (1/1), 932 bytes | 932.00 KiB/s, done.
From https://github.com/yeezyyoba/amharic-sentiment-analysis
 * branch            main       -> FETCH_HEAD
   6ee45c1..15de8b4  main       -> origin/main
Updating 6ee45c1..15de8b4
Fast-forward
 .gitignore                         |   2 --
 data/processed/test_clean.parquet  | Bin 0 -> 431021 bytes
 data/processed/test_raw.parquet    | Bin 0 -> 227883 bytes
 data/processed/train_clean.parquet | Bin 0 -> 1234285 bytes
 data/processed/train_raw.parquet   | Bin 0 -> 650123 bytes
 data/processed/val_clean.parquet   | Bin 0 -> 313471 bytes
 data/processed/val_raw.parquet     | Bin 0 -> 166200 bytes
 7 files changed, 2 deletions(-)
 create mode 100644 data/processed/test_clean.parquet
 create mode 100644 data/processed/test_raw.parquet
 create mode 100644 data/processed/train_clean.

In [13]:
import re
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorWithPadding

# ── Load clean parquets ────────────────────────────────────────
train_df = pd.read_parquet("data/processed/train_clean.parquet")
val_df   = pd.read_parquet("data/processed/val_clean.parquet")
test_df  = pd.read_parquet("data/processed/test_clean.parquet")

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")
print(f"Columns: {train_df.columns.tolist()}")

# ── Label map ──────────────────────────────────────────────────
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL  = {0: "negative", 1: "neutral", 2: "positive"}

# ── Tokenize ───────────────────────────────────────────────────
MODEL_NAME = "Davlan/afro-xlmr-base"
MAX_LENGTH = 72
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(df):
    df = df.copy()
    df["labels"] = df["sentiment"].map(LABEL2ID)
    return Dataset.from_pandas(df[["clean_tweet", "labels"]], preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["clean_tweet"], truncation=True,
                     max_length=MAX_LENGTH, padding=False)

print("\nTokenizing...")
tokenized = DatasetDict({
    "train":      make_dataset(train_df).map(tokenize, batched=True),
    "validation": make_dataset(val_df).map(tokenize, batched=True),
    "test":       make_dataset(test_df).map(tokenize, batched=True)
})

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized)
print("\n✅ Ready for training.")

Train: 5,968
Val:   1,496
Test:  1,999
Columns: ['tweet', 'clean_tweet', 'label', 'sentiment']


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...


Tokenizing...


Map:   0%|          | 0/5968 [00:00<?, ? examples/s]

Map:   0%|          | 0/1496 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['clean_tweet', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 5968
    })
    validation: Dataset({
        features: ['clean_tweet', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1496
    })
    test: Dataset({
        features: ['clean_tweet', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1999
    })
})

✅ Ready for training.


In [16]:
import torch
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import evaluate

# ── Metrics ────────────────────────────────────────────────────
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

# ── Training function ──────────────────────────────────────────
def train_one_seed(seed):
    print(f"\n{'='*50}")
    print(f"  Training with seed {seed}")
    print(f"{'='*50}")

    torch.manual_seed(seed)
    np.random.seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True
    )

    output_dir = f"{DRIVE_SAVE_PATH}/seed_{seed}"

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=2e-5,
        warmup_steps=100,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        seed=seed,
        fp16=True,
        logging_steps=50,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    # Evaluate on test set
    test_results = trainer.evaluate(tokenized["test"])
    test_f1 = test_results["eval_f1"]

    print(f"\nSeed {seed} — Test Macro F1: {test_f1:.4f}")

    # Save model to Google Drive
    trainer.save_model(f"{output_dir}/best_model")
    print(f"Model saved to {output_dir}/best_model")

    return test_f1

# ── Run all 3 seeds ────────────────────────────────────────────
SEEDS = [42, 123, 456]
results = []

for seed in SEEDS:
    f1 = train_one_seed(seed)
    results.append(f1)

# ── Final results ──────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"  FINAL RESULTS")
print(f"{'='*50}")
for seed, f1 in zip(SEEDS, results):
    print(f"  Seed {seed}: Macro F1 = {f1:.4f}")
print(f"\n  Mean F1:  {np.mean(results):.4f}")
print(f"  Std F1:   {np.std(results):.4f}")
print(f"\n  Result for paper: {np.mean(results):.4f} ± {np.std(results):.4f}")


  Training with seed 42


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,0.881292,0.814299,0.548353
2,0.795126,0.794809,0.602688
3,0.728377,0.814804,0.598505
4,0.664789,0.865482,0.589070


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1
0.557013,1.058940,5,0.545500



Seed 456 — Test Macro F1: 0.5455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/amharic_sentiment_models/seed_456/best_model

  FINAL RESULTS
  Seed 42: Macro F1 = 0.5620
  Seed 123: Macro F1 = 0.5553
  Seed 456: Macro F1 = 0.5455

  Mean F1:  0.5543
  Std F1:   0.0068

  Result for paper: 0.5543 ± 0.0068


In [17]:
# Save results to file
import json

paper_results = {
    "baseline": {
        "model": "TF-IDF + Logistic Regression",
        "macro_f1": 0.4640
    },
    "transformer": {
        "model": "Afro-XLM-R (Davlan/afro-xlmr-base)",
        "seeds": [42, 123, 456],
        "per_seed_f1": [0.5620, 0.5553, 0.5455],
        "mean_f1": 0.5543,
        "std_f1": 0.0068,
        "paper_result": "0.5543 ± 0.0068",
        "improvement_over_baseline": "+9.03 points"
    }
}

with open("/content/drive/MyDrive/amharic_sentiment_models/paper_results.json", "w") as f:
    json.dump(paper_results, f, indent=2)

print("Results saved to Google Drive!")
print(json.dumps(paper_results, indent=2))

Results saved to Google Drive!
{
  "baseline": {
    "model": "TF-IDF + Logistic Regression",
    "macro_f1": 0.464
  },
  "transformer": {
    "model": "Afro-XLM-R (Davlan/afro-xlmr-base)",
    "seeds": [
      42,
      123,
      456
    ],
    "per_seed_f1": [
      0.562,
      0.5553,
      0.5455
    ],
    "mean_f1": 0.5543,
    "std_f1": 0.0068,
    "paper_result": "0.5543 \u00b1 0.0068",
    "improvement_over_baseline": "+9.03 points"
  }
}
